# Adım 6: Makine Öğrenmesi — Çoklu Model Karşılaştırma

**Problem tipi:** Regresyon — NYC Taxi ücret tahmini (`fare_amount`)

| Model | Algoritma |
|-------|-----------|
| 1 | Linear Regression |
| 2 | Ridge Regression (GLR) |
| 3 | Decision Tree Regressor |
| 4 | Random Forest Regressor |
| 5 | Gradient Boosted Trees (GBT) |

**Değerlendirme metrikleri:** RMSE · MAE · R² · Residual Analizi · Feature Importance  
**MLflow:** Tüm denemeler `NYC_Taxi_Fare_Prediction` experiment altında loglanır.

## 1. Kurulum

In [ ]:
import subprocess, sys
pkgs = ["pandas", "numpy", "scikit-learn", "mlflow", "matplotlib", "seaborn", "pyarrow==15.0.2"]
for pkg in pkgs:
    r = subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                       capture_output=True, text=True)
    print(f"{'✅' if r.returncode==0 else '❌'} {pkg}")

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

PROJECT_ROOT  = Path(os.getcwd()).parent
GOLD_PATH     = PROJECT_ROOT / "data/delta/gold/fare_features"
ENG_PATH      = PROJECT_ROOT / "data/delta/gold/engineered_features"
PARQUET_PATH  = PROJECT_ROOT / "data/raw/yellow_tripdata_2023-01.parquet"
PLOTS_DIR     = PROJECT_ROOT / "data/ml_plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_URI    = os.getenv("MLFLOW_TRACKING_URI", "http://localhost:5000")
print(f"MLflow URI: {MLFLOW_URI}")

## 2. Veri Yükleme & Hazırlık

In [ ]:
import pyarrow.parquet as pq

SAMPLE = 30_000

def load_df():
    # Engineered features
    eng_file = ENG_PATH / "engineered_features.parquet"
    if eng_file.exists():
        return pd.read_parquet(eng_file), "Engineered Features"
    # Gold Delta
    if list(GOLD_PATH.rglob("*.parquet")):
        return pd.read_parquet(GOLD_PATH), "Gold Delta"
    # Ham Parquet fallback
    raw = pq.read_table(PARQUET_PATH).to_pandas()
    raw["pickup_datetime"] = pd.to_datetime(raw["tpep_pickup_datetime"], errors="coerce")
    raw["pickup_hour"]     = raw["pickup_datetime"].dt.hour
    raw["pickup_day_of_week"] = raw["pickup_datetime"].dt.dayofweek + 1
    raw = raw[
        (raw["trip_distance"] > 0) & (raw["trip_distance"] <= 100) &
        (raw["fare_amount"]   > 0) & (raw["fare_amount"]   <= 500)
    ].rename(columns={"fare_amount": "label"})
    return raw, "Ham Parquet"

df_raw, kaynak = load_df()
if len(df_raw) > SAMPLE:
    df_raw = df_raw.sample(SAMPLE, random_state=42).reset_index(drop=True)

print(f"Kaynak : {kaynak}")
print(f"Boyut  : {df_raw.shape}")

## 3. Feature Seçimi & Encoding

In [ ]:
label_col = "label" if "label" in df_raw.columns else "fare_amount"

feature_candidates = [
    "trip_distance", "passenger_count", "pickup_hour",
    "pickup_day_of_week", "PULocationID", "DOLocationID",
    "RatecodeID", "payment_type",
    "fare_per_mile", "is_rush_hour", "is_weekend",
    "cross_borough", "is_airport_trip",
    "pickup_borough", "dropoff_borough",
    "distance_bin", "passenger_group",
]

df = df_raw.copy()
# Kategorik sütunları encode et
for col in ["pickup_borough", "dropoff_borough", "distance_bin", "passenger_group"]:
    if col in df.columns:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

features = [c for c in feature_candidates if c in df.columns]
X = df[features].fillna(0)
y = df[label_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Kullanılan feature sayısı : {len(features)}")
print(f"Eğitim seti               : {len(X_train):,}")
print(f"Test seti                 : {len(X_test):,}")
print(f"\nFeatures: {features}")

## 4. Model Tanımları

In [ ]:
MODELS = {
    "Linear_Regression":      LinearRegression(),
    "Ridge_Regression":       Ridge(alpha=1.0),
    "Decision_Tree":          DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random_Forest":          RandomForestRegressor(n_estimators=50, max_depth=10,
                                                    random_state=42, n_jobs=-1),
    "Gradient_Boosted_Trees": GradientBoostingRegressor(n_estimators=50,
                                                        learning_rate=0.1,
                                                        max_depth=5, random_state=42),
}

print(f"{'Model':<30} {'Parametreler'}")
print("-" * 65)
for ad, m in MODELS.items():
    params = {k: v for k, v in m.get_params().items() if v is not None}
    kisa = {k: v for i, (k, v) in enumerate(params.items()) if i < 3}
    print(f"  {ad:<28} {kisa}")

## 5. MLflow Bağlantı Kontrolü

In [ ]:
import urllib.request

mlflow_aktif = False
try:
    urllib.request.urlopen(MLFLOW_URI, timeout=3)
    mlflow.set_tracking_uri(MLFLOW_URI)
    mlflow_aktif = True
    print(f"✅ MLflow erişilebilir: {MLFLOW_URI}")
except Exception:
    mlflow.set_tracking_uri("mlruns")
    print(f"⚠️  MLflow sunucusu yok → yerel 'mlruns/' kullanılıyor")

mlflow.set_experiment("NYC_Taxi_Fare_Prediction")
print("Experiment: NYC_Taxi_Fare_Prediction")

## 6. Model Eğitimi & MLflow Loglama

In [ ]:
sonuclar = []

for model_adi, model in MODELS.items():
    print(f"\n→ {model_adi} eğitiliyor...")

    with mlflow.start_run(run_name=model_adi):
        # Eğitim
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        # Metrikler
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae  = mean_absolute_error(y_test, y_pred)
        r2   = r2_score(y_test, y_pred)
        residuals = y_test.values - y_pred

        # MLflow — Parametreler
        mlflow.log_param("model_type", model_adi)
        mlflow.log_param("n_features", len(features))
        mlflow.log_param("train_size", len(X_train))
        for pk, pv in list(model.get_params().items())[:5]:
            mlflow.log_param(pk, pv)

        # MLflow — Metrikler
        mlflow.log_metric("rmse",    round(rmse, 4))
        mlflow.log_metric("mae",     round(mae, 4))
        mlflow.log_metric("r2_score",round(r2, 4))

        # Grafik 1: Gerçek vs Tahmin
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.scatter(y_test, y_pred, alpha=0.3, s=10, color="#3498db")
        mn, mx = y_test.min(), y_test.max()
        ax.plot([mn, mx], [mn, mx], "r--", lw=2, label="İdeal")
        ax.set_xlabel("Gerçek Ücret ($)")
        ax.set_ylabel("Tahmin ($)")
        ax.set_title(f"{model_adi}\nRMSE={rmse:.2f} | R²={r2:.3f}")
        ax.legend()
        scatter_path = PLOTS_DIR / f"{model_adi}_scatter.png"
        plt.savefig(scatter_path, bbox_inches="tight")
        mlflow.log_artifact(str(scatter_path))
        plt.close()

        # Grafik 2: Residual Dağılımı
        fig, ax = plt.subplots(figsize=(6, 4))
        sns.histplot(residuals, bins=60, kde=True, color="#9b59b6", ax=ax)
        ax.axvline(0, color="red", lw=2)
        ax.set_xlabel("Hata ($)")
        ax.set_title(f"{model_adi} — Residual Dağılımı")
        res_path = PLOTS_DIR / f"{model_adi}_residual.png"
        plt.savefig(res_path, bbox_inches="tight")
        mlflow.log_artifact(str(res_path))
        plt.close()

        # Grafik 3: Feature Importance (ağaç tabanlı)
        if hasattr(model, "feature_importances_"):
            fi = pd.Series(model.feature_importances_, index=features).nlargest(10)
            fig, ax = plt.subplots(figsize=(7, 4))
            fi.sort_values().plot.barh(ax=ax, color="#1abc9c", alpha=0.85)
            ax.set_title(f"{model_adi} — Top-10 Feature Importance")
            fi_path = PLOTS_DIR / f"{model_adi}_fi.png"
            plt.savefig(fi_path, bbox_inches="tight")
            mlflow.log_artifact(str(fi_path))
            plt.close()

        # Modeli kaydet
        mlflow.sklearn.log_model(model, "model")

        sonuclar.append({
            "Model": model_adi,
            "RMSE":  round(rmse, 3),
            "MAE":   round(mae, 3),
            "R²":    round(r2, 4),
        })
        print(f"   ✅ RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")

print("\n✅ Tüm modeller tamamlandı!")

## 7. Model Karşılaştırma Tablosu

In [ ]:
sonuc_df = pd.DataFrame(sonuclar).sort_values("RMSE")
sonuc_df["Sıra"] = range(1, len(sonuc_df)+1)
sonuc_df = sonuc_df.set_index("Sıra")

print("=" * 60)
print("   MODEL KARŞILAŞTIRMA TABLOSU (RMSE'ye göre)")
print("=" * 60)
print(sonuc_df.to_string())
print("=" * 60)
en_iyi = sonuc_df.iloc[0]
print(f"\n🏆 En iyi model: {en_iyi['Model']}")
print(f"   RMSE={en_iyi['RMSE']}  MAE={en_iyi['MAE']}  R²={en_iyi['R²']}")

## 8. Metrik Karşılaştırma Grafikleri

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
renkler = ["#e74c3c","#e67e22","#f1c40f","#2ecc71","#3498db"]

metrikler = [
    ("RMSE", "RMSE (↓ daha iyi)",  True),
    ("MAE",  "MAE (↓ daha iyi)",   True),
    ("R²",   "R² Skoru (↑ daha iyi)", False),
]

for ax, (metrik, baslik, asagi) in zip(axes, metrikler):
    siralama = sonuc_df.sort_values(metrik, ascending=asagi)
    bars = ax.barh(siralama["Model"], siralama[metrik],
                   color=renkler[:len(siralama)], alpha=0.85)
    ax.set_title(baslik)
    ax.set_xlabel(metrik)
    for bar, val in zip(bars, siralama[metrik]):
        ax.text(bar.get_width() + bar.get_width()*0.01, bar.get_y() + bar.get_height()/2,
                f"{val:.3f}", va="center", fontsize=8)

plt.suptitle("Model Karşılaştırma — Regresyon Metrikleri", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model_comparison.png", bbox_inches="tight")
plt.show()

## 9. En İyi Model — Feature Importance

In [ ]:
en_iyi_adi  = sonuc_df.iloc[0]["Model"]
en_iyi_mdl  = MODELS[en_iyi_adi]

if hasattr(en_iyi_mdl, "feature_importances_"):
    fi = pd.Series(en_iyi_mdl.feature_importances_, index=features).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    colors  = ["#e74c3c" if v == fi.max() else "#3498db" for v in fi]
    fi.plot.barh(ax=ax, color=colors, alpha=0.85)
    ax.set_title(f"{en_iyi_adi} — Tüm Feature Importance")
    ax.set_xlabel("Önem Skoru")
    plt.tight_layout()
    plt.show()

    print("\nTop-5 En Etkili Özellikler:")
    for feat, imp in fi.nlargest(5).items():
        print(f"  {feat:<30} {imp:.4f} ({imp*100:.1f}%)")
else:
    # Linear model için katsayılar
    coef = pd.Series(en_iyi_mdl.coef_, index=features).abs().sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    coef.plot.barh(ax=ax, color="#9b59b6", alpha=0.85)
    ax.set_title(f"{en_iyi_adi} — Katsayı Büyüklükleri")
    plt.tight_layout()
    plt.show()

## 10. En İyi Model — Residual Analizi

In [ ]:
y_pred_best = en_iyi_mdl.predict(X_test)
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residual histogram
sns.histplot(residuals, bins=60, kde=True, color="#9b59b6", ax=axes[0])
axes[0].axvline(0, color="red", lw=2)
axes[0].set_title(f"{en_iyi_adi}\nResidual Dağılımı")
axes[0].set_xlabel("Hata ($)")

# Residual vs Fitted
axes[1].scatter(y_pred_best, residuals, alpha=0.2, s=8, color="#e67e22")
axes[1].axhline(0, color="red", lw=2)
axes[1].set_xlabel("Tahmin Edilen Değer ($)")
axes[1].set_ylabel("Residual ($)")
axes[1].set_title("Residual vs Fitted")

plt.suptitle("Residual Analizi", fontsize=13)
plt.tight_layout()
plt.show()

print(f"Residual istatistikleri:")
print(f"  Ortalama hata : {residuals.mean():.3f}$")
print(f"  Std hata      : {residuals.std():.3f}$")
print(f"  Maks hata     : {residuals.max():.2f}$")
print(f"  Min hata      : {residuals.min():.2f}$")

## 11. MLflow UI Erişimi

In [ ]:
print("MLflow deneme sonuçları için:")
print(f"  → Tarayıcıda açın: {MLFLOW_URI}")
print(f"  → Experiment: NYC_Taxi_Fare_Prediction")
print()

if not mlflow_aktif:
    print("Yerel MLflow UI başlatmak için:")
    print(f"  mlflow ui --backend-store-uri {PROJECT_ROOT / 'mlruns'} --port 5001")
    print(f"  → http://localhost:5001")

## 12. Özet & Teslim Edilecekler

| # | Teslim | Durum |
|---|--------|-------|
| 1 | 5 regresyon modeli eğitimi | ✅ |
| 2 | RMSE, MAE, R² metrikleri | ✅ |
| 3 | Residual analizi | ✅ (Bölüm 10) |
| 4 | Feature Importance analizi | ✅ (Bölüm 9) |
| 5 | MLflow loglama (parametreler, metrikler, modeller) | ✅ (Bölüm 6) |
| 6 | Gerçek vs Tahmin scatter grafikleri | ✅ |
| 7 | Model karşılaştırma grafikleri | ✅ (Bölüm 8) |

**Grafik çıktıları:** `data/ml_plots/`

---
> **Tüm adımlar tamamlandı!** Pipeline: Kafka → Bronze → Silver → Gold → Feature Engineering → ML Modelleri → MLflow